# Colab Part C: fine-tune Laya on the liquidation decisions

**J26-SE-325, Component 1.** Laya (`convaiinnovations/laya`) answers typed questions with
calibrated probabilities. This notebook fine-tunes it on the two-stage question set in
`decision/questions.py`:

* **Stage 1 (analysis):** technical trend / momentum / support, fundamental health /
  valuation / earnings / distress, each from its own compact state.
* **Stage 2 (decision):** per holding sell now / spread / defer / exclude and downside risk;
  per portfolio the urgency and risk tolerance.

Labels come from **what actually happened next** (`decision/labels.py`), built only from
information available on each date, and split **by date**: the test set is the most recent
20% of dates. The encoder stays frozen, so the update that comes home is a few MB.

**Prerequisites on Drive:** Part A's `bars.parquet` and Part D's `fundamentals_backfill.parquet`.
**Runtime:** T4 GPU. **Resumable:** every stage is saved on Drive and skipped when present.

In [ ]:
# === BOOTSTRAP (same contract as colab_finetune.ipynb) ================================
# Clone or update the branch, chdir into the component, install PIP_PKGS. Safe to re-run.
REPO_URL = "https://github.com/SE-Y4S1/J26-SE-325.git"
BRANCH = "Nivakaran"  # the component is NOT on main
SUBDIR = "backend/Portfolio-Optimization"
PIP_PKGS = "laya pypdf yfinance pandas-ta-openbb pyyaml requests pydantic pydantic-settings scikit-fuzzy"

import importlib, os, subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
OWN_PACKAGES = {"agent", "configs", "data", "evaluation", "experiments", "features",
                "forecasting", "fundamentals", "news", "optimization", "decision", "service"}


def find_root(start):
    for base in (start, *start.parents):
        if (base / "configs/universe.yaml").exists() and (base / "fundamentals").exists():
            return base
    return None


# In Colab always fetch: a re-run starts INSIDE the checkout, so find_root
# succeeded and pushed fixes never reached a live runtime.
root = None if IN_COLAB else find_root(Path.cwd())
if root is None:
    if not IN_COLAB:
        raise SystemExit("cd into backend/Portfolio-Optimization first.")
    checkout = Path("/content/J26-SE-325")
    if (checkout / SUBDIR).exists():
        if subprocess.run(["git", "-C", str(checkout), "fetch", "--depth", "1", "origin", BRANCH]).returncode == 0:
            subprocess.run(["git", "-C", str(checkout), "reset", "--hard", "FETCH_HEAD"])
    else:
        if subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(checkout)]).returncode:
            from getpass import getpass
            token = getpass("Clone failed. GitHub token (repo scope), blank to abort: ").strip()
            if not token:
                raise SystemExit("Cannot reach " + REPO_URL)
            subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                            REPO_URL.replace("https://", "https://" + token + "@"), str(checkout)], check=True)
    root = checkout / SUBDIR

os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
for _name in [n for n in sys.modules if n.split(".")[0] in OWN_PACKAGES]:
    del sys.modules[_name]
importlib.invalidate_caches()
if IN_COLAB and PIP_PKGS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + PIP_PKGS.split(), check=True)
print("OK root=" + str(root))

In [ ]:
import torch

print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Runtime > Change runtime type > T4 GPU"
import laya

print("laya", laya.__version__)

In [ ]:
from pathlib import Path

OUT = Path("artifacts/laya_training")
BARS = Path("artifacts/hybrid_training/bars.parquet")
BACKFILL = Path("artifacts/backfill/fundamentals_backfill.parquet")
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = Path("/content/drive/MyDrive/j26_laya")
    BARS = Path("/content/drive/MyDrive/j26_hybrid/bars.parquet")
    BACKFILL = Path("/content/drive/MyDrive/j26_se_325_backfill/fundamentals_backfill.parquet")
OUT.mkdir(parents=True, exist_ok=True)
assert BARS.exists(), "Run Part A first (it writes bars.parquet)"
print("fundamentals backfill:", BACKFILL if BACKFILL.exists() else "NOT FOUND -- fundamental questions get no rows")

## Smoke run

A few hundred rows and one epoch: validates the dataset against Laya's own training code and
the whole pipeline in minutes.

In [ ]:
from experiments.train_laya import main

args = ["--out", str(OUT / "smoke"), "--bars", str(BARS), "--scenarios", "200",
        "--sample-every", "40", "--epochs", "1"]
if BACKFILL.exists():
    args += ["--backfill", str(BACKFILL)]
main(args)

## Full run

About 20k scenarios plus every symbol's Stage 1 rows, two fine-tunes, then evaluation.
Several hours on one T4; re-run this cell after a disconnect.

In [ ]:
args = ["--out", str(OUT), "--bars", str(BARS)]
if BACKFILL.exists():
    args += ["--backfill", str(BACKFILL)]
main(args)

## RQ5: zero-shot vs fine-tuned, on dates after the cutoff

Accuracy of the most likely answer against the realised outcome, Brier score (lower is
better) and mean confidence, per question. A fine-tuned model that does not beat zero-shot on
a question is reported as such.

In [ ]:
import pandas as pd

table = pd.read_csv(OUT / "evaluation.csv")
table.pivot(index="question", columns="model", values=["accuracy", "brier"]).round(3)

## Bring the head home

Then, on the local machine (the base checkpoint must be cached once:
`huggingface-cli download convaiinnovations/laya`):

```bash
python -m decision.checkpoint --install laya_head.zip
```

In [ ]:
if IN_COLAB:
    from google.colab import files
    files.download(str(OUT / "laya_head.zip"))